In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Install Dependencies

In [2]:
import pandas as pd
import os as os
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/media/hassan-elkersh/OS/Users/User/OneDrive - Cairo University - Students/Graduation project/AutoPrepAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Dataset

In [3]:
path = os.path.expanduser("Input/train.csv") 
df = pd.read_csv(path)
df.head()

,answers,query,finalpassage
0,"Kids who are bipolar, in their manic stages, v...",why do children get aggressive,"At the same time, despite claiming the review ..."
1,"Equifax, transunion and experian.",which credit bureau is used the most for auto ...,Best Answer: both of those answers are wrong. ...
2,"Women eat at least 1,200 calories daily and me...",what is the minimum healthy calorie intake,Safe Intakes. If you’re not supervised by a me...
3,Because Caffeine increases the stress hormone ...,why is coffee making gain weight,Is coffee making you fat? If you are overweigh...
4,Kent County,"what county is grand rapids, mi in","Located in Grand Rapids, Michigan, the 61st Di..."


### Select Only the `query` Column

In [4]:
df = df[['query']]
df.head()

,query
0,why do children get aggressive
1,which credit bureau is used the most for auto ...
2,what is the minimum healthy calorie intake
3,why is coffee making gain weight
4,"what county is grand rapids, mi in"


### Get Number of Rows

In [5]:
len(df)

120000

### Display the First 20 Rows of the DataFrame

In [6]:
df.head(20)

,query
0,why do children get aggressive
1,which credit bureau is used the most for auto ...
2,what is the minimum healthy calorie intake
3,why is coffee making gain weight
4,"what county is grand rapids, mi in"
5,is bing rewards available in india
6,what temperature does it need to be to pour co...
7,how to clean stain on travertine tile
8,what state is alameda in
9,salary sacrifice contributions age limit


### Take a Random Sample

In [7]:
sample = df.sample(10000, random_state=42)

### Load Model

In [8]:
# model = SentenceTransformer("all-MiniLM-L6-v2")
model = SentenceTransformer("all-mpnet-base-v2")


### Embeddings

In [9]:
embeddings = model.encode(sample['query'].tolist(), show_progress_bar=True, convert_to_numpy=True, batch_size=64)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches: 100%|██████████| 157/157 [00:04<00:00, 38.39it/s]


### Cosine Similarity

In [10]:
sim_matrix = cosine_similarity(embeddings)

### Detect Semantics

In [11]:
threshold = 0.8
for i in range(len(sample)):
    for j in range(i+1, len(sample)):
        if sim_matrix[i][j] > threshold:
            print(f"Pair found with similarity {sim_matrix[i][j]:.2f}:")
            print(f"Query 1: {sample.iloc[i]['query']}")
            print(f"Query 2: {sample.iloc[j]['query']}")
            print()

Pair found with similarity 0.98:
Query 1: The name Olmec means _____.
Query 2: the name olmec means

Pair found with similarity 0.93:
Query 1: define marketing procurement
Query 2: procurement marketing definition

Pair found with similarity 0.85:
Query 1: taklamakan desert definition
Query 2: where is taklimakan desert located

Pair found with similarity 0.85:
Query 1: what are good sources of calcium
Query 2: what food gives you calcium

Pair found with similarity 0.95:
Query 1: what is average salary for pharmacist
Query 2: what is pharmacists salary

Pair found with similarity 0.91:
Query 1: what is average salary for pharmacist
Query 2: how much does a pharmacist make

Pair found with similarity 0.83:
Query 1: how long can you keep eggs in the fridge
Query 2: how long do fresh eggs keep

Pair found with similarity 0.97:
Query 1: rita name meaning
Query 2: meaning of the name rita

Pair found with similarity 0.94:
Query 1: garage cost to build
Query 2: how much does it cost to buil

### Remove Duplicates

In [ ]:
to_drop = set()

threshold = 0.8
for i in range(len(sample)):
    for j in range(i + 1, len(sample)):
        if sim_matrix[i][j] > threshold:
            to_drop.add(j)

deduped_sample = sample.drop(sample.index[list(to_drop)]).reset_index(drop=True)

print(f"Original size: {len(sample)}")
print(f"After deduplication: {len(deduped_sample)}")


Original size: 10000
After deduplication: 9760
Data After deduplication:                                                   query
0               how to select multiple items in outlook
1     can an attorney terminate a contingency fee ag...
2                 sidvokodvo average weather conditions
3                       what do molecules do constantly
4                    what is a group of muskrats called
...                                                 ...
9755                            postage cost for letter
9756                      define spread plate technique
9757                          average ocean temperature
9758                where do atlantic silversides spawn
9759                 what does elizabethan england mean

[9760 rows x 1 columns]
